# CAPM & Fama-French Factor Model — CAC 40
### Beta, Alpha, SMB, HML appliqués aux valeurs françaises

Ce notebook applique les deux grands modèles d'évaluation des actifs financiers
aux composantes du **CAC 40**, avec des données réelles téléchargées via **yfinance**
et les facteurs de risque européens de la **bibliothèque Ken French**.

**Plan :**
1. Univers CAC 40 — données, performance normalisée, statistiques
2. Matrices de corrélation — entre titres et entre secteurs (heatmaps)
3. CAPM — beta, alpha de Jensen, droite de marché (SML)
4. Fama-French 3 facteurs — chargements Mkt-RF, SMB, HML par titre et par secteur
5. Synthèse — classements alpha et beta

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import yfinance as yf
import statsmodels.api as sm
import warnings

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")
plt.rcParams.update({"figure.dpi": 120, "font.size": 11, "axes.titlesize": 13})

## 1. Univers CAC 40 — Téléchargement des données

In [ ]:
# ── CAC 40 : ~30 valeurs représentatives avec leur secteur ──────────────────
CAC40 = {
    # Luxe
    "MC.PA":   "Luxe",          # LVMH
    "RMS.PA":  "Luxe",          # Hermès
    "KER.PA":  "Luxe",          # Kering
    # Consommation
    "OR.PA":   "Consommation",  # L'Oréal
    "BN.PA":   "Consommation",  # Danone
    "RI.PA":   "Consommation",  # Pernod Ricard
    # Finance
    "BNP.PA":  "Finance",       # BNP Paribas
    "ACA.PA":  "Finance",       # Crédit Agricole
    "GLE.PA":  "Finance",       # Société Générale
    "CS.PA":   "Finance",       # AXA
    # Énergie
    "TTE.PA":  "Énergie",       # TotalEnergies
    "ENGI.PA": "Énergie",       # Engie
    # Santé
    "SAN.PA":  "Santé",         # Sanofi
    "EL.PA":   "Santé",         # EssilorLuxottica
    # Industrie
    "AIR.PA":  "Industrie",     # Airbus
    "SAF.PA":  "Industrie",     # Safran
    "SU.PA":   "Industrie",     # Schneider Electric
    "DG.PA":   "Industrie",     # Vinci
    "LR.PA":   "Industrie",     # Legrand
    # Technologie
    "CAP.PA":  "Technologie",   # Capgemini
    "DSY.PA":  "Technologie",   # Dassault Systèmes
    "STM.PA":  "Technologie",   # STMicroelectronics
    # Matériaux
    "MT.PA":   "Matériaux",     # ArcelorMittal
    "SGO.PA":  "Matériaux",     # Saint-Gobain
    # Automobile
    "RNO.PA":  "Automobile",    # Renault
    # Services aux collectivités
    "VIE.PA":  "Services",      # Veolia
    # Télécom / Communication
    "ORA.PA":  "Télécom",       # Orange
    "PUB.PA":  "Communication", # Publicis
}

START = "2019-01-01"
END   = "2024-12-31"

tickers = list(CAC40.keys())
sectors = pd.Series(CAC40, name="Secteur")

# Téléchargement des cours + indice CAC 40 comme proxy de marché
raw = yf.download(tickers + ["^FCHI"], start=START, end=END,
                  auto_adjust=True, progress=False)["Close"]
raw = raw.rename(columns={"^FCHI": "CAC40"})
raw = raw.ffill().dropna(how="all")

# Supprimer les colonnes avec trop de NaN
raw = raw.loc[:, raw.isnull().mean() < 0.10]
available = [t for t in tickers if t in raw.columns]
sectors   = sectors[sectors.index.isin(available)]

prices = raw[available + ["CAC40"]].dropna()

print(f"Période : {prices.index[0].date()} → {prices.index[-1].date()}")
print(f"Titres  : {len(available)} valeurs CAC 40 + indice")
print(f"Lignes  : {len(prices)} jours de cotation")
prices.tail(3)

## 2. Performance normalisée (base 100)

In [ ]:
returns = prices.pct_change().dropna()
stock_ret  = returns[available]
market_ret = returns["CAC40"]

# Normalisation
base = prices / prices.iloc[0] * 100
SECTOR_COLORS = {
    "Luxe": "#c0392b", "Finance": "#2980b9", "Industrie": "#27ae60",
    "Technologie": "#8e44ad", "Énergie": "#e67e22", "Santé": "#16a085",
    "Consommation": "#f39c12", "Matériaux": "#7f8c8d",
    "Automobile": "#d35400", "Services": "#1abc9c",
    "Télécom": "#2c3e50", "Communication": "#e74c3c",
}

fig, ax = plt.subplots(figsize=(14, 6))
for ticker in available:
    sect  = CAC40.get(ticker, "Autre")
    color = SECTOR_COLORS.get(sect, "gray")
    ax.plot(base.index, base[ticker], linewidth=1.0,
            alpha=0.55, color=color)

# CAC 40 en noir épais
ax.plot(base.index, base["CAC40"], color="black", linewidth=2.5,
        label="CAC 40 (indice)", zorder=5)
ax.axhline(100, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Performance normalisée des valeurs CAC 40 (base 100 = jan. 2019)")
ax.set_ylabel("Indice")

# Légende manuelle par secteur
from matplotlib.lines import Line2D
legend_elements = [Line2D([0],[0], color=c, linewidth=2, label=s)
                   for s, c in SECTOR_COLORS.items() if s in sectors.values]
legend_elements.append(Line2D([0],[0], color="black", linewidth=2.5, label="CAC 40"))
ax.legend(handles=legend_elements, ncol=3, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 3. Statistiques descriptives

In [ ]:
ann_ret = stock_ret.mean() * 252
ann_vol = stock_ret.std()  * np.sqrt(252)
sharpe  = ann_ret / ann_vol

stats = pd.DataFrame({
    "Rendement annualisé (%)": (ann_ret * 100).round(2),
    "Volatilité annualisée (%)": (ann_vol * 100).round(2),
    "Sharpe Ratio":  sharpe.round(2),
    "Secteur": sectors,
}).sort_values("Sharpe Ratio", ascending=False)

# Top 5 / Flop 5
print("── Top 5 Sharpe ──────────────────────────────────────────")
print(stats.head(5).to_string())
print("\n── Flop 5 Sharpe ─────────────────────────────────────────")
print(stats.tail(5).to_string())

## 4. Matrice de corrélation — tous les titres

In [ ]:
corr = stock_ret.corr()

fig, ax = plt.subplots(figsize=(14, 11))

# Trier par secteur pour faire apparaître les blocs
order = sectors.sort_values().index.tolist()
order = [t for t in order if t in corr.index]
corr_sorted = corr.loc[order, order]

# Labels courts (ticker sans .PA)
short_labels = [t.replace(".PA", "") for t in order]

mask = np.zeros_like(corr_sorted, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = True  # masque triangle supérieur

sns.heatmap(
    corr_sorted, mask=mask,
    cmap="RdYlGn", vmin=-0.2, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 7},
    linewidths=0.4, linecolor="white",
    xticklabels=short_labels, yticklabels=short_labels,
    ax=ax, cbar_kws={"shrink": 0.6}
)

ax.set_xticklabels(short_labels, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(short_labels, rotation=0, fontsize=8)
ax.set_title("Matrice de corrélation — Rendements journaliers CAC 40\n(trié par secteur)", pad=12)

# Lignes de séparation entre secteurs
sect_order = sectors[order].values
boundaries = [0]
for i in range(1, len(sect_order)):
    if sect_order[i] != sect_order[i-1]:
        boundaries.append(i)
boundaries.append(len(sect_order))
for b in boundaries[1:-1]:
    ax.axhline(b, color="navy", linewidth=1.5)
    ax.axvline(b, color="navy", linewidth=1.5)

plt.tight_layout()
plt.show()

## 5. Corrélation inter-sectorielle

In [ ]:
# Rendement journalier moyen par secteur
sect_daily = stock_ret.copy()
sect_daily.columns = sectors[sect_daily.columns]
sector_avg = sect_daily.T.groupby(level=0).mean().T

sector_corr = sector_avg.corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap symétrique complète
sns.heatmap(
    sector_corr, ax=axes[0],
    cmap="RdYlGn", vmin=-0.2, vmax=1,
    annot=True, fmt=".2f", annot_kws={"size": 9},
    linewidths=0.8, square=True,
    cbar_kws={"shrink": 0.7}
)
axes[0].set_title("Corrélation entre secteurs\n(rendements journaliers moyens)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha="right")
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)

# Clustermap de performance cumulée par secteur
cumul_sector = (1 + sector_avg).cumprod()
for col in cumul_sector.columns:
    axes[1].plot(cumul_sector.index, cumul_sector[col],
                 linewidth=2, label=col,
                 color=SECTOR_COLORS.get(col, "gray"))
axes[1].axhline(1, color="black", linestyle="--", linewidth=0.8)
axes[1].set_title("Performance cumulée par secteur (base 1 = jan. 2019)")
axes[1].set_ylabel("Croissance de 1 €")
axes[1].legend(ncol=2, fontsize=8)

plt.tight_layout()
plt.show()

## 6. Corrélation avec le CAC 40 — Rolling 90 jours

In [ ]:
# Corrélation glissante de chaque secteur avec le CAC 40
roll_corr = pd.DataFrame({
    sect: sector_avg[sect].rolling(90).corr(market_ret)
    for sect in sector_avg.columns
}).dropna()

fig, ax = plt.subplots(figsize=(14, 5))
for sect in roll_corr.columns:
    ax.plot(roll_corr.index, roll_corr[sect],
            linewidth=1.5, label=sect,
            color=SECTOR_COLORS.get(sect, "gray"))
ax.axhline(1, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.axhline(0, color="black", linewidth=0.6)
ax.set_title("Corrélation glissante 90 jours de chaque secteur avec le CAC 40")
ax.set_ylabel("Corrélation avec le marché")
ax.legend(ncol=3, fontsize=8)
plt.tight_layout()
plt.show()

## 7. CAPM — Modèle d'Évaluation des Actifs Financiers

$$r_i - r_f = \alpha_i + \beta_i (r_m - r_f) + \varepsilon_i$$

- **β > 1** : titre plus volatile que le marché (amplificateur de cycle)
- **β < 1** : titre défensif (amortit les chocs de marché)
- **α > 0** : surperformance résiduelle non expliquée par le risque systématique

In [ ]:
RF_ANNUAL = 0.03          # taux sans risque approximatif (OAT 10 ans ~3%)
rf_daily  = RF_ANNUAL / 252

excess_market = market_ret - rf_daily

capm_results = {}
for ticker in available:
    excess_stock = stock_ret[ticker] - rf_daily
    X = sm.add_constant(excess_market)
    model = sm.OLS(excess_stock, X).fit()
    capm_results[ticker] = {
        "alpha_ann": model.params["const"] * 252,
        "beta":      model.params["CAC40"],
        "r2":        model.rsquared,
        "p_alpha":   model.pvalues["const"],
        "p_beta":    model.pvalues["CAC40"],
        "t_alpha":   model.tvalues["const"],
    }

capm_df = pd.DataFrame(capm_results).T
capm_df["secteur"]   = sectors
capm_df["ticker_short"] = capm_df.index.str.replace(".PA", "", regex=False)
capm_df = capm_df.sort_values("beta")

print("── CAPM results ──────────────────────────────────────────────────────")
display_cols = ["secteur", "beta", "alpha_ann", "r2", "p_alpha"]
print(capm_df[display_cols].round(4).to_string())

## 8. Security Market Line (SML)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

# SML théorique
beta_range = np.linspace(capm_df["beta"].min() - 0.1,
                          capm_df["beta"].max() + 0.1, 100)
market_premium = float(ann_ret["^FCHI"]) if "^FCHI" in ann_ret else (market_ret.mean() * 252 - RF_ANNUAL)
sml_ret = RF_ANNUAL + beta_range * market_premium

ax.plot(beta_range, sml_ret * 100, color="black", linewidth=1.8,
        linestyle="--", label="SML (CAPM théorique)", zorder=2)

# Points par secteur
for sect in sectors.unique():
    mask = capm_df["secteur"] == sect
    sub  = capm_df[mask]
    act_ret = ann_ret[sub.index] * 100
    ax.scatter(sub["beta"], act_ret,
               color=SECTOR_COLORS.get(sect, "gray"),
               s=90, edgecolors="white", linewidths=0.5,
               zorder=3, label=sect)
    for _, row in sub.iterrows():
        ax.annotate(row["ticker_short"],
                    (row["beta"], ann_ret[row.name] * 100),
                    fontsize=7.5, ha="left", va="bottom",
                    xytext=(3, 3), textcoords="offset points")

ax.axhline(RF_ANNUAL * 100, color="gray", linestyle=":", linewidth=1,
           label=f"Taux sans risque ({RF_ANNUAL:.1%})")
ax.axvline(1, color="gray", linestyle=":", linewidth=1)
ax.set_title("Security Market Line — CAC 40\nAu-dessus de la SML = alpha positif")
ax.set_xlabel("Beta (risque systématique β)")
ax.set_ylabel("Rendement annualisé réel (%)")
ax.legend(ncol=2, fontsize=8, loc="upper left")
plt.tight_layout()
plt.show()

## 9. Heatmaps CAPM — Beta et Alpha par secteur

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── A : Beta moyen par secteur ──────────────────────────────────────────────
beta_by_sector = capm_df.groupby("secteur")["beta"].mean().sort_values(ascending=False)
colors_bar = [SECTOR_COLORS.get(s, "gray") for s in beta_by_sector.index]

axes[0].barh(beta_by_sector.index, beta_by_sector.values,
             color=colors_bar, edgecolor="white")
axes[0].axvline(1, color="black", linestyle="--", linewidth=1.2)
axes[0].set_title("Beta moyen par secteur")
axes[0].set_xlabel("β moyen")

# ── B : Alpha annualisé moyen par secteur ────────────────────────────────────
alpha_by_sector = capm_df.groupby("secteur")["alpha_ann"].mean().sort_values(ascending=False)
colors_alpha = ["green" if v > 0 else "crimson" for v in alpha_by_sector.values]

axes[1].barh(alpha_by_sector.index, alpha_by_sector.values * 100,
             color=colors_alpha, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Alpha annualisé moyen par secteur (%)")
axes[1].set_xlabel("α annualisé (%)")

# ── C : Heatmap Beta × Secteur (chaque titre) ────────────────────────────────
pivot = capm_df.pivot_table(index="secteur", columns="ticker_short",
                             values="beta", aggfunc="first")
pivot = pivot.reindex(sorted(pivot.index))

sns.heatmap(pivot, ax=axes[2], cmap="RdYlGn_r", center=1,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.5, cbar_kws={"shrink": 0.7},
            mask=pivot.isnull())
axes[2].set_title("Beta par titre et par secteur")
axes[2].set_xlabel("")
axes[2].set_ylabel("")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45, ha="right", fontsize=7)

plt.tight_layout()
plt.show()

## 10. Classement Alpha — Qui crée (ou détruit) de la valeur ?

In [ ]:
top_alpha = capm_df.sort_values("alpha_ann", ascending=False).head(10)
bot_alpha = capm_df.sort_values("alpha_ann").head(5)

fig, ax = plt.subplots(figsize=(12, 6))
combined = pd.concat([top_alpha, bot_alpha]).drop_duplicates()
combined = combined.sort_values("alpha_ann")

bar_colors = [
    SECTOR_COLORS.get(combined.loc[i, "secteur"], "gray")
    for i in combined.index
]
bars = ax.barh(combined["ticker_short"], combined["alpha_ann"] * 100,
               color=bar_colors, edgecolor="white")
ax.axvline(0, color="black", linewidth=1)

for bar, val in zip(bars, combined["alpha_ann"] * 100):
    ax.text(val + (0.1 if val >= 0 else -0.1), bar.get_y() + bar.get_height()/2,
            f"{val:+.1f}%", va="center", ha="left" if val >= 0 else "right",
            fontsize=8)

ax.set_title("Alpha de Jensen annualisé — CAC 40\n(CAPM, rf=3%, marché=CAC 40)")
ax.set_xlabel("Alpha annualisé (%)")
plt.tight_layout()
plt.show()

## 11. Fama-French 3 Facteurs

Le modèle de Fama-French ajoute deux facteurs au CAPM :

$$r_i - r_f = \alpha_i + \beta_i^{Mkt}(r_m - r_f) + \beta_i^{SMB} \cdot SMB + \beta_i^{HML} \cdot HML + \varepsilon_i$$

| Facteur | Définition | Intuition |
|---------|------------|-----------|
| **Mkt-RF** | Prime de risque marché | Risque systématique |
| **SMB** | Small Minus Big | Les petites caps surperforment historiquement |
| **HML** | High Minus Low (book-to-market) | Les valeurs "value" surperforment les "growth" |

Nous téléchargeons les facteurs **européens quotidiens** de la bibliothèque Ken French.

In [ ]:
# Téléchargement des facteurs Fama-French européens
try:
    import pandas_datareader.data as web
    ff_raw = web.DataReader("Europe_3_Factors_Daily", "famafrench",
                             start=START, end=END)
    ff = ff_raw[0].copy()
    ff.index = pd.to_datetime(ff.index)
    ff = ff / 100   # convertir % → décimales
    ff.columns = ["Mkt-RF", "SMB", "HML", "RF"]
    print(f"✅ Facteurs FF européens téléchargés : {len(ff)} observations")
    print(ff.tail(3).round(5))
    FF_SOURCE = "Fama-French Europe Daily (Ken French Library)"

except Exception as e:
    print(f"⚠️  Téléchargement FF échoué : {e}")
    print("   → Construction de proxies SMB et HML depuis les données CAC 40...")

    # ── Proxy SMB : petites caps (vol élevée) - grandes caps (vol faible) ──
    annual_vol_rank = stock_ret.std().rank()
    small_tickers = annual_vol_rank.nlargest(8).index.tolist()
    large_tickers = annual_vol_rank.nsmallest(8).index.tolist()
    smb_proxy = stock_ret[small_tickers].mean(axis=1) - stock_ret[large_tickers].mean(axis=1)

    # ── Proxy HML : "value" (faible momentum) - "growth" (fort momentum) ──
    momentum = (prices[available].iloc[-1] / prices[available].iloc[0] - 1)
    value_tickers  = momentum.nsmallest(8).index.tolist()
    growth_tickers = momentum.nlargest(8).index.tolist()
    hml_proxy = stock_ret[value_tickers].mean(axis=1) - stock_ret[growth_tickers].mean(axis=1)

    ff = pd.DataFrame({
        "Mkt-RF": market_ret - rf_daily,
        "SMB":    smb_proxy,
        "HML":    hml_proxy,
        "RF":     rf_daily,
    }, index=market_ret.index)
    FF_SOURCE = "Proxies construits depuis les données CAC 40"

In [ ]:
# Alignement des indices
common_idx = stock_ret.index.intersection(ff.index)
stock_ff   = stock_ret.loc[common_idx]
ff_aligned = ff.loc[common_idx]

print(f"Source facteurs : {FF_SOURCE}")
print(f"Observations alignées : {len(common_idx)} jours")

# ── Régression Fama-French pour chaque titre ────────────────────────────────
ff_results = {}
for ticker in available:
    y = stock_ff[ticker] - ff_aligned["RF"]
    X = sm.add_constant(ff_aligned[["Mkt-RF", "SMB", "HML"]])
    model = sm.OLS(y, X).fit()
    ff_results[ticker] = {
        "alpha_ann":  model.params["const"] * 252,
        "beta_mkt":   model.params["Mkt-RF"],
        "beta_smb":   model.params["SMB"],
        "beta_hml":   model.params["HML"],
        "r2":         model.rsquared,
        "r2_adj":     model.rsquared_adj,
        "p_alpha":    model.pvalues["const"],
    }

ff_df = pd.DataFrame(ff_results).T
ff_df["secteur"]      = sectors
ff_df["ticker_short"] = ff_df.index.str.replace(".PA", "", regex=False)

print("\n── Fama-French factor loadings (extrait) ──────────────────────────────")
print(ff_df[["secteur","alpha_ann","beta_mkt","beta_smb","beta_hml","r2_adj"]]
      .round(4).sort_values("alpha_ann", ascending=False).head(10).to_string())

## 12. Heatmap des chargements Fama-French par secteur

In [ ]:
# Chargements moyens par secteur
loadings_sector = ff_df.groupby("secteur")[["beta_mkt","beta_smb","beta_hml","alpha_ann"]].mean()
loadings_sector.columns = ["β Mkt-RF", "β SMB", "β HML", "α annualisé"]
loadings_sector["α annualisé"] *= 100   # en %

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── A : Heatmap des chargements moyens ──────────────────────────────────────
sns.heatmap(
    loadings_sector.drop(columns="α annualisé"),
    ax=axes[0], cmap="RdYlGn", center=0,
    annot=True, fmt=".3f", annot_kws={"size": 10},
    linewidths=0.8, cbar_kws={"shrink": 0.7}
)
axes[0].set_title("Chargements Fama-French moyens par secteur\n(β Mkt-RF, β SMB, β HML)")
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)
axes[0].set_yticklabels(axes[0].get_yticklabels(), rotation=0)

# ── B : Alpha FF vs Alpha CAPM par secteur ───────────────────────────────────
alpha_compare = pd.DataFrame({
    "Alpha CAPM (%)":  capm_df.groupby("secteur")["alpha_ann"].mean() * 100,
    "Alpha FF (%)":    loadings_sector["α annualisé"],
}).sort_values("Alpha CAPM (%)", ascending=True)

x = np.arange(len(alpha_compare))
w = 0.35
bars1 = axes[1].barh(x - w/2, alpha_compare["Alpha CAPM (%)"], w,
                      label="Alpha CAPM", color="steelblue", edgecolor="white")
bars2 = axes[1].barh(x + w/2, alpha_compare["Alpha FF (%)"],   w,
                      label="Alpha Fama-French", color="coral", edgecolor="white")
axes[1].set_yticks(x)
axes[1].set_yticklabels(alpha_compare.index, fontsize=9)
axes[1].axvline(0, color="black", linewidth=0.8)
axes[1].set_title("Alpha CAPM vs Alpha Fama-French par secteur (%)")
axes[1].set_xlabel("Alpha annualisé (%)")
axes[1].legend()

plt.tight_layout()
plt.show()

## 13. Heatmap complète : chargements individuels (tous les titres × 3 facteurs)

In [ ]:
# Pivot : titre × facteur
loadings_all = ff_df[["ticker_short","secteur","beta_mkt","beta_smb","beta_hml"]].copy()
loadings_all = loadings_all.sort_values(["secteur","ticker_short"])
loadings_matrix = loadings_all.set_index("ticker_short")[["beta_mkt","beta_smb","beta_hml"]]
loadings_matrix.columns = ["β Mkt-RF", "β SMB", "β HML"]

fig, ax = plt.subplots(figsize=(8, max(7, len(loadings_matrix) * 0.45)))
sns.heatmap(
    loadings_matrix, ax=ax,
    cmap="RdYlGn", center=0,
    annot=True, fmt=".2f", annot_kws={"size": 8},
    linewidths=0.5, cbar_kws={"shrink": 0.6}
)
ax.set_title("Chargements Fama-French par titre\n(trié par secteur)")
ax.set_xticklabels(["β Marché", "β Petites caps (SMB)", "β Valeur (HML)"], rotation=0)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0, fontsize=8)

# Séparateurs de secteur
sect_sorted = loadings_all["secteur"].values
boundaries  = [i for i in range(1, len(sect_sorted))
               if sect_sorted[i] != sect_sorted[i-1]]
for b in boundaries:
    ax.axhline(b, color="navy", linewidth=1.5)

plt.tight_layout()
plt.show()

## 14. Synthèse — Tableau de bord final

In [ ]:
# CAPM vs FF : R² ajusté moyen par secteur
r2_compare = pd.DataFrame({
    "R² CAPM":          capm_df.groupby("secteur")["r2"].mean(),
    "R² Fama-French":   ff_df.groupby("secteur")["r2_adj"].mean(),
})

fig = plt.figure(figsize=(18, 12))
gs  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)

# ── 1 : Beta par titre ───────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
cd  = capm_df.sort_values("beta")
colors_b = [SECTOR_COLORS.get(r["secteur"], "gray") for _, r in cd.iterrows()]
ax1.barh(cd["ticker_short"], cd["beta"], color=colors_b, edgecolor="white", height=0.7)
ax1.axvline(1, color="black", linestyle="--", linewidth=1)
ax1.set_title("1. Beta (CAPM) par titre")
ax1.set_xlabel("β")
ax1.tick_params(labelsize=7)

# ── 2 : Alpha FF par titre ───────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
fd  = ff_df.sort_values("alpha_ann")
colors_a = ["green" if v > 0 else "crimson" for v in fd["alpha_ann"]]
ax2.barh(fd["ticker_short"], fd["alpha_ann"] * 100, color=colors_a,
         edgecolor="white", height=0.7)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_title("2. Alpha Fama-French annualisé (%)")
ax2.set_xlabel("α (%)")
ax2.tick_params(labelsize=7)

# ── 3 : R² CAPM vs FF ────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
x3 = np.arange(len(r2_compare))
ax3.bar(x3 - 0.2, r2_compare["R² CAPM"] * 100, 0.4,
        label="CAPM", color="steelblue", edgecolor="white")
ax3.bar(x3 + 0.2, r2_compare["R² Fama-French"] * 100, 0.4,
        label="Fama-French", color="coral", edgecolor="white")
ax3.set_xticks(x3)
ax3.set_xticklabels(r2_compare.index, rotation=45, ha="right", fontsize=8)
ax3.set_title("3. R² ajusté moyen par secteur (%)")
ax3.set_ylabel("R² (%)")
ax3.legend(fontsize=8)

# ── 4 : Corrélation inter-sectorielle (heatmap) ──────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
sns.heatmap(sector_corr, ax=ax4, cmap="RdYlGn", vmin=-0.2, vmax=1,
            annot=True, fmt=".2f", annot_kws={"size": 7},
            linewidths=0.5, square=True, cbar=False)
ax4.set_title("4. Corrélation inter-sectorielle")
ax4.set_xticklabels(ax4.get_xticklabels(), rotation=45, ha="right", fontsize=7)
ax4.set_yticklabels(ax4.get_yticklabels(), rotation=0, fontsize=7)

# ── 5 : Chargements FF par secteur (heatmap) ─────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
sns.heatmap(loadings_sector.drop(columns="α annualisé"),
            ax=ax5, cmap="RdYlGn", center=0,
            annot=True, fmt=".2f", annot_kws={"size": 8},
            linewidths=0.5, cbar=False)
ax5.set_title("5. Chargements FF par secteur")
ax5.set_xticklabels(["β Mkt", "β SMB", "β HML"], rotation=0, fontsize=9)
ax5.set_yticklabels(ax5.get_yticklabels(), rotation=0, fontsize=8)

# ── 6 : Performance cumulée par secteur ──────────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
for col in cumul_sector.columns:
    ax6.plot(cumul_sector.index, cumul_sector[col],
             linewidth=1.8, label=col, color=SECTOR_COLORS.get(col, "gray"))
ax6.axhline(1, color="black", linestyle="--", linewidth=0.8)
ax6.set_title("6. Performance cumulée par secteur")
ax6.set_ylabel("Croissance de 1 €")
ax6.legend(ncol=2, fontsize=7)

fig.suptitle("CAC 40 — Tableau de bord CAPM & Fama-French",
             fontsize=16, fontweight="bold", y=1.01)
plt.show()